# Run:ai

A practical refresher on **Run:ai** — a commercial GPU orchestration platform for Kubernetes, now part of NVIDIA. Run:ai sits on top of a Kubernetes cluster and replaces (or augments) the default scheduler to make GPUs a first-class, shareable, quota-governed resource for AI/ML teams.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Workload Manifests](#manifests)
7. [Fractional GPUs & Quotas](#fractions)
8. [Use Cases](#use-cases)
9. [Best Practices](#best-practices)
10. [Common Pitfalls](#pitfalls)
11. [Performance Optimization](#performance)
12. [Production Deployment](#deployment)
13. [Monitoring and Observability](#monitoring)
14. [Troubleshooting](#troubleshooting)
15. [Comparison with Alternatives](#comparison)
16. [Resources](#resources)

## Introduction <a id="introduction"></a>

Run:ai provides a **GPU scheduling and orchestration layer for Kubernetes**. NVIDIA acquired Run:ai in 2024 and in 2025 open-sourced the core scheduler as the **KAI Scheduler** (Apache 2.0), while continuing to sell the full Run:ai platform (control plane, UI, quotas, RBAC, reporting) as part of NVIDIA AI Enterprise.

### What is it?

- A **batch/AI scheduler** (`runai-scheduler`) that you run instead of, or alongside, the default `kube-scheduler`. It understands GPUs, gang scheduling, fair-share, and preemption.
- A **fractional-GPU runtime** that lets several pods share a single physical GPU (by GPU-memory slice or time-slicing) instead of the all-or-nothing `nvidia.com/gpu: 1`.
- A **multi-tenant control plane**: Projects and Departments map onto namespaces and carry GPU/CPU/memory quotas, plus a web UI, CLI (`runai`), RBAC, and usage dashboards.

### Why use it?

- **Higher GPU utilization.** Fractional GPUs and time-slicing pack many small notebooks/inference pods onto one GPU; idle quota is loaned out via over-quota fair-share.
- **Fair multi-tenancy.** Each team gets a *guaranteed* quota but can burst into unused capacity, and is preempted back down when owners reclaim it.
- **Gang scheduling for distributed jobs.** All workers of a multi-GPU PyTorch/MPI job start together or not at all — no half-scheduled jobs holding GPUs hostage.
- **Self-service + governance.** Data scientists submit via CLI/UI; platform teams keep quota, priority, and policy guardrails.

### When to use it (and when not)

- **Reach for it** when you run a shared GPU cluster across multiple teams, want fractional sharing, and value a vendor-supported product with a UI and chargeback reporting.
- **Skip it** for a single-team cluster where stock `kube-scheduler` + the NVIDIA device plugin (or open-source Kueue/Volcano) is enough, or when you specifically want to avoid a commercial dependency — in which case the open-source **KAI Scheduler** gives you the scheduling core without the platform.

## Key Features <a id="key-features"></a>

| Feature | What it does | Why it matters |
|---|---|---|
| **Fractional GPUs** | Allocate a slice of GPU memory (e.g. `--gpu-memory 4G`) or a fraction (`--gpu 0.5`) instead of a whole device. | Run 2–8 light workloads per GPU; dramatically higher utilization for notebooks/inference. |
| **Time-slicing & MIG** | Software time-slicing on any GPU, plus support for hardware MIG partitions on A100/H100. | Choose between cheap soft isolation and strong hardware isolation. |
| **Guaranteed quota + over-quota** | Projects get a guaranteed GPU quota and can burst into idle cluster capacity. | Teams are never starved, yet idle GPUs don't sit empty. |
| **Gang scheduling** | All pods of a distributed job are scheduled atomically. | No deadlocks where partial jobs hold GPUs and block everyone. |
| **Preemption & priority** | Build/interactive vs train vs inference priority classes; over-quota work is preemptible. | Interactive work stays snappy; reclaimed quota returns to owners. |
| **Node pools** | Group nodes (by GPU type, zone, etc.) and target workloads at them. | Send big jobs to H100 pools, dev work to T4 pools. |
| **Projects & Departments** | Hierarchical tenancy mapped to namespaces with RBAC. | Clean per-team isolation, quota, and reporting. |
| **Dashboards & reporting** | GPU allocation/utilization, queue depth, historical consumption. | Capacity planning and cost chargeback. |

## Architecture Overview <a id="architecture"></a>

Run:ai is split into a **control plane** and one or more **cluster** installations:

```
                +-----------------------------+
                |        Control Plane         |   (SaaS or self-hosted)
                |  Web UI · Auth/RBAC · API    |
                |  Quota & policy · Dashboards |
                +--------------+--------------+
                               | (agent / API)
        +----------------------+-----------------------+
        |              Kubernetes cluster              |
        |                                              |
        |  runai-scheduler   <-- schedules GPU pods    |
        |  runai-operator    <-- reconciles workloads  |
        |  GPU fraction runtime / device plugin        |
        |  cluster agent     <-- syncs to control plane|
        |                                              |
        |  Project namespaces: runai-<project> ...     |
        +----------------------------------------------+
```

Key pieces:

- **`runai-scheduler`** — the GPU-aware scheduler. Workloads opt in via `schedulerName: runai-scheduler` (the CLI and CRDs set this for you).
- **`runai-operator`** — reconciles Run:ai workload CRDs (workspaces, training, distributed, inference) into the underlying Kubernetes objects.
- **Fractional-GPU runtime** — intercepts CUDA so a pod sees only its slice of GPU memory; combined with the NVIDIA device plugin.
- **Projects = namespaces.** A Project named `team-a` lives in namespace `runai-team-a`; quota and RBAC are attached to it.

From an ML engineer's seat you keep submitting Kubernetes-shaped work — you just target a Project and request whole or fractional GPUs.

## Installation <a id="installation"></a>

Run:ai is a **commercial product** installed by platform/infra teams, not via `pip`. At a high level:

1. **Control plane** — use Run:ai SaaS, or self-host with Helm:
   ```bash
   helm repo add runai https://run-ai-charts.storage.googleapis.com
   helm upgrade -i runai-backend runai/control-plane -n runai-backend --create-namespace
   ```
2. **Cluster install** — register the cluster in the control plane, then install the cluster components (scheduler, operator, GPU runtime):
   ```bash
   helm upgrade -i runai-cluster runai/runai-cluster -n runai \
     --create-namespace --set controlPlane.url=<your-cp-url> --set cluster.uid=<uid>
   ```
3. **Prereqs** — a working Kubernetes cluster with the **NVIDIA GPU Operator** (drivers + device plugin) already installed, plus an ingress controller and cert-manager.
4. **CLI** — download the `runai` CLI from the control-plane (`Settings → Researcher CLI`) and authenticate:
   ```bash
   runai login
   runai config cluster <cluster-name>
   ```

Exact chart names/versions change between releases — always follow the version-matched docs for your deployment. The cell below just records that there is nothing to `pip install`.

In [ ]:
# Run:ai is NOT a Python package — it is a Kubernetes platform driven by `kubectl`,
# the `runai` CLI, and the web UI. There is nothing to `pip install`.
#
# This notebook is a conceptual + manifest refresher. The cells below GENERATE the
# YAML/CLI you would run against a real Run:ai cluster; they don't talk to a cluster.

tools = {
    "kubectl": "apply/get/describe the underlying Kubernetes objects",
    "runai (CLI)": "submit & manage workloads, projects, quota",
    "helm": "install the control plane and cluster components",
    "web UI": "dashboards, RBAC, quota, reporting",
}
for name, role in tools.items():
    print(f"{name:14s} -> {role}")

## Basic Usage <a id="basic-usage"></a>

Day-to-day you use the `runai` CLI. The modern CLI (v2.x) is organized by **workload type**; the classic CLI used a flat `runai submit`. Common commands:

```bash
# Auth & context
runai login
runai project list
runai project set team-a            # pick your working project/namespace

# Interactive workspace (a Jupyter/dev pod) requesting half a GPU
runai workspace submit dev-box \
  --image jupyter/scipy-notebook --gpu-request 0.5 --interactive

# A training job on 1 full GPU
runai training submit train-resnet \
  --image nvcr.io/nvidia/pytorch:24.05-py3 --gpu-request 1 \
  --command -- python train.py --epochs 20

# A distributed PyTorch job: 1 master + 3 workers, 1 GPU each (gang scheduled)
runai training pytorch submit ddp-job \
  --image nvcr.io/nvidia/pytorch:24.05-py3 --workers 3 --gpu-request 1 \
  --command -- torchrun --nproc_per_node=1 train.py

# Observe
runai training list
runai training logs train-resnet -f
runai training describe train-resnet
runai training delete train-resnet
```

Classic-CLI equivalent (still seen in older clusters):

```bash
runai submit train-resnet -i pytorch:24.05-py3 -g 1 -p team-a -- python train.py
runai list jobs
runai logs train-resnet
```

In [ ]:
# Build a `runai` submit command programmatically (e.g. from an experiment config).
# Demonstrates the typical flags without needing a live cluster.

def runai_training_submit(name, image, gpus, project, command, workers=0):
    parts = ["runai", "training"]
    parts += ["pytorch", "submit", name] if workers else ["submit", name]
    parts += ["--project", project, "--image", image, "--gpu-request", str(gpus)]
    if workers:
        parts += ["--workers", str(workers)]
    parts += ["--command", "--", *command]
    return " ".join(parts)

print(runai_training_submit(
    "sweep-lr-0.01", "nvcr.io/nvidia/pytorch:24.05-py3",
    gpus=0.5, project="team-a", command=["python", "train.py", "--lr", "0.01"]))
print()
print(runai_training_submit(
    "ddp-8gpu", "nvcr.io/nvidia/pytorch:24.05-py3",
    gpus=1, project="team-a", workers=7,
    command=["torchrun", "--nproc_per_node=1", "train.py"]))

## Workload Manifests <a id="manifests"></a>

Under the hood every workload is a Kubernetes object that opts into the Run:ai scheduler via `schedulerName: runai-scheduler` and lives in a Project namespace (`runai-<project>`). You can submit raw manifests with `kubectl apply` when you need GitOps or fine control.

A plain training **Job** targeting the Run:ai scheduler and requesting one GPU:

```yaml
apiVersion: batch/v1
kind: Job
metadata:
  name: train-resnet
  namespace: runai-team-a          # Project 'team-a'
spec:
  template:
    metadata:
      labels:
        project: team-a            # used for quota accounting
    spec:
      schedulerName: runai-scheduler
      restartPolicy: Never
      containers:
        - name: train
          image: nvcr.io/nvidia/pytorch:24.05-py3
          command: ["python", "train.py"]
          resources:
            limits:
              nvidia.com/gpu: 1
```

Run:ai also ships higher-level CRDs (`run.ai` API group) that the operator expands — e.g. `TrainingWorkload`, `InteractiveWorkload`, `DistributedWorkload`, `InferenceWorkload`. These carry Run:ai-native fields (fractions, parallelism, autoscaling) that a stock Job can't express. The CLI generates these for you; the raw Job above is the lowest-common-denominator form that's easy to reason about.

In [ ]:
# Generate a Run:ai-scheduled Kubernetes Job manifest in pure Python (no deps),
# including a fractional-GPU request expressed as an annotation.

def runai_job_yaml(name, project, image, command, gpu_fraction=None, gpus=0):
    annotations = ""
    limits = ""
    if gpu_fraction is not None:
        # Fractional GPU: request a slice; Run:ai enforces the memory limit.
        annotations = (
            "\n      annotations:\n"
            f'        gpu-fraction: "{gpu_fraction}"'
        )
    elif gpus:
        limits = f"\n            limits:\n              nvidia.com/gpu: {gpus}"
    cmd = ", ".join(f'"{c}"' for c in command)
    return f"""apiVersion: batch/v1
kind: Job
metadata:
  name: {name}
  namespace: runai-{project}
spec:
  template:
    metadata:
      labels:
        project: {project}{annotations}
    spec:
      schedulerName: runai-scheduler
      restartPolicy: Never
      containers:
        - name: {name}
          image: {image}
          command: [{cmd}]
          resources:{limits or ' {}'}
"""

print("# Whole-GPU training job:")
print(runai_job_yaml("train-resnet", "team-a",
                     "nvcr.io/nvidia/pytorch:24.05-py3",
                     ["python", "train.py"], gpus=1))
print("# Fractional-GPU inference pod (40% of one GPU):")
print(runai_job_yaml("serve-bert", "team-a",
                     "my-registry/bert-serve:latest",
                     ["python", "serve.py"], gpu_fraction=0.4))

## Fractional GPUs & Quotas <a id="fractions"></a>

Two ideas do most of the work in Run:ai:

**1. Fractional GPUs.** Instead of the binary `nvidia.com/gpu: 1`, you ask for a fraction (`--gpu-request 0.5`) or an explicit GPU-memory size (`--gpu-memory 4G`). Run:ai caps the pod's visible GPU memory and time-slices compute. Modes:
- **Fraction** — share by a percentage of memory/compute (soft isolation, any GPU).
- **GPU memory** — request an absolute amount of VRAM.
- **MIG** — carve hardware-isolated slices on A100/H100 (strong isolation).

**2. Quota & fair-share.** Each Project has a **guaranteed (deserved) GPU quota**. Beyond that it can run **over-quota** by borrowing idle GPUs from other projects. When an owning project needs its quota back, Run:ai **preempts** over-quota, lower-priority work. Priority classes (interactive `build`, `train`, `inference`) decide what gets preempted first — inference and guaranteed work are protected; over-quota training is the first to yield.

The cell below models the over-quota/fair-share math so the intuition sticks.

In [ ]:
# Toy model of Run:ai guaranteed-quota + fair-share over-quota allocation.
# Each project has a guaranteed quota and a current demand; idle GPUs from
# under-using projects are loaned out, split by deserved quota (fair-share).

total_gpus = 16
projects = {
    # name: (guaranteed_quota, demand)
    "team-a": (8, 12),   # wants to burst over quota
    "team-b": (4, 1),    # under-using -> lends capacity
    "team-c": (4, 6),    # also wants to burst
}

# 1) Everyone first gets min(demand, guaranteed).
alloc = {p: min(d, q) for p, (q, d) in projects.items()}
free = total_gpus - sum(alloc.values())

# 2) Distribute the free pool to projects still wanting more, weighted by quota.
wanters = {p: projects[p][1] - alloc[p] for p in projects if projects[p][1] > alloc[p]}
weight_total = sum(projects[p][0] for p in wanters)
for p in wanters:
    share = round(free * projects[p][0] / weight_total)
    alloc[p] += min(share, wanters[p])

print(f"Cluster GPUs: {total_gpus}  |  free pool to lend: {free}\n")
for p, (q, d) in projects.items():
    tag = "OVER-QUOTA" if alloc[p] > q else "within quota"
    print(f"{p}: quota={q} demand={d:>2} -> allocated={alloc[p]:>2}  ({tag})")
print(f"\nTotal allocated: {sum(alloc.values())}/{total_gpus}")

## Use Cases <a id="use-cases"></a>

- **Shared GPU platform for many ML teams** — one cluster, per-team Projects with quota, RBAC, and chargeback reporting.
- **Notebook/IDE fleets** — dozens of interactive sessions packed onto a handful of GPUs via fractions instead of one GPU per idle notebook.
- **Cost-efficient inference** — small models served at fractional-GPU granularity, scaled with the inference workload type.
- **Large distributed training** — multi-node DDP/Megatron jobs that need gang scheduling and topology-aware placement.
- **Burst experimentation** — researchers borrow idle cluster GPUs over-quota for hyperparameter sweeps, preempted when owners return.

## Best Practices <a id="best-practices"></a>

1. **Right-size GPU requests.** Use fractions (`--gpu-request 0.3`) or `--gpu-memory` for notebooks and small inference; reserve whole/MIG GPUs for training that actually saturates the device.
2. **Pick the correct workload type.** `workspace` (interactive) vs `training` (batch, preemptible-friendly) vs `inference` (long-lived, protected). The type drives priority and preemption behavior.
3. **Lean on over-quota, but expect preemption.** Make training jobs checkpoint regularly so a preempt-and-requeue costs minutes, not hours.
4. **Use node pools intentionally.** Route heavy jobs to H100 pools and dev work to cheaper T4/L4 pools to avoid hoarding premium GPUs.
5. **Keep images lean and cached.** Long pulls dominate short jobs; pre-pull or use a registry mirror near the cluster.
6. **Codify submissions.** Generate manifests/CLI from config (as in the cells above) so experiments are reproducible and GitOps-friendly.

## Common Pitfalls <a id="pitfalls"></a>

1. **Treating fractional GPUs as hardware-isolated.** Fractions/time-slicing isolate *memory*, not compute perfectly — noisy neighbors can slow each other. Use MIG when you need hard isolation.
2. **Quota surprises.** Jobs sit `Pending` because the Project is at guaranteed quota and no idle GPUs are free to lend. Check `runai project list` / the dashboard before blaming the scheduler.
3. **Non-preemptible assumptions.** Over-quota training *will* be preempted. Jobs without checkpointing lose progress.
4. **Forgetting `schedulerName`.** A raw manifest without `schedulerName: runai-scheduler` is handled by the default scheduler and bypasses quota/fairness entirely.
5. **Wrong namespace.** Submitting into a non-Project namespace means no quota accounting and likely no GPU access. Always target `runai-<project>`.
6. **MIG vs fraction mismatch.** Requesting a fraction on a MIG-only node (or vice versa) leaves pods unschedulable.

## Performance Optimization <a id="performance"></a>

- **Profile shared vs exclusive.** Benchmark your workload at `--gpu-request 1`, `0.5`, and `0.25` — find the point where throughput-per-GPU stops improving.
- **Match sharing mode to workload.** Time-slicing suits bursty/interactive pods; MIG suits steady inference needing predictable latency.
- **Use gang + topology awareness** for distributed training so all workers land on well-connected nodes (NVLink/IB), avoiding straggler-induced stalls.
- **Tune priority/preemption** so latency-sensitive inference is never preempted while batch training absorbs the churn.
- **Co-design node pools** (GPU type, count, network) with infra so the scheduler has the right shapes to place work efficiently.

## Production Deployment <a id="deployment"></a>

- **Owned by platform/infra**, usually with NVIDIA/Run:ai support. The control plane can be SaaS or self-hosted; one control plane can manage many clusters.
- **Integrations** — wire RBAC to your IdP (OIDC/SAML), connect storage (PVCs, object stores) and networking (ingress, IB/NVLink), and the container registry.
- **Upgrades** — keep control-plane and cluster component versions compatible; roll the scheduler/operator with the cluster on a maintenance cadence.
- **Policies & guardrails** — set default/limit GPU requests, allowed images, and idle-timeout policies so self-service stays safe.

## Monitoring and Observability <a id="monitoring"></a>

- **Run:ai dashboards** — GPU allocation vs utilization, queue depth, per-Project/Department consumption, and historical trends for chargeback.
- **Prometheus/Grafana** — Run:ai and the NVIDIA DCGM exporter expose GPU metrics (utilization, memory, temperature) you can scrape into your own stack.
- **`runai` CLI** — `runai training describe`, `runai training logs -f`, and `runai project list` for quick at-the-terminal status.
- **Watch allocation-vs-utilization gap** — high *allocation* but low *utilization* is the classic signal that fractions are mis-sized or jobs are idle.

## Troubleshooting <a id="troubleshooting"></a>

| Symptom | Likely cause | What to check |
|---|---|---|
| Job stuck `Pending` | At quota with no idle GPUs to lend, or no node matches the request | `runai training describe <job>`, `kubectl describe pod`, project quota |
| Job preempted/requeued | Owning project reclaimed over-quota capacity | Priority/class of the job; add checkpointing |
| Pod scheduled by wrong scheduler | Missing `schedulerName: runai-scheduler` | Pod spec; resubmit via CLI/CRD |
| `CUDA out of memory` on a fraction | Requested GPU-memory slice too small | Increase `--gpu-memory` / fraction |
| No GPUs visible in pod | NVIDIA GPU Operator / device plugin not healthy | `kubectl get pods -n gpu-operator`, node labels |
| Control-plane data stale | Cluster agent not syncing | Cluster connection status in the UI; agent pod logs |

## Comparison with Alternatives <a id="comparison"></a>

| Aspect | **Run:ai** | **KAI Scheduler** (OSS) | **Kueue / Volcano / YuniKorn** (OSS) | **Cloud batch** (AWS Batch, GKE) |
|---|---|---|---|---|
| Type | Commercial platform | Open-source scheduler (NVIDIA) | Open-source schedulers | Managed cloud service |
| Fractional GPUs | Yes (fraction/memory/MIG) | Yes (the Run:ai core) | Via time-slicing/MIG plumbing you wire up | Limited / instance-bound |
| Quota & fair-share | Rich, with UI & RBAC | Quota/fair-share core | Yes (Kueue/Volcano) | Cloud quotas |
| Gang scheduling | Yes | Yes | Volcano/Kueue: yes | Service-dependent |
| UI / reporting / support | Full product + vendor support | None (DIY) | None (DIY) | Cloud console |
| Best for | Enterprises wanting a turnkey GPU platform | Teams wanting Run:ai's scheduler without the platform | DIY platform builders | Cloud-native, single-cloud shops |

**Choose Run:ai** when you want a vendor-supported, multi-tenant GPU platform with a UI, quotas, and chargeback. **Choose KAI Scheduler** when you want Run:ai's open-source scheduling brain without the commercial platform. **Choose Kueue/Volcano** for a fully open-source DIY stack, or **cloud batch** when you're all-in on one cloud and don't need fractional sharing.

## Resources <a id="resources"></a>

### Official Documentation

- Run:ai docs — https://docs.run.ai/
- NVIDIA Run:ai product page — https://www.nvidia.com/en-us/software/run-ai/
- KAI Scheduler (open-source core) — https://github.com/NVIDIA/KAI-Scheduler

### Guides & Background

- Run:ai fractional GPU / GPU sharing concepts — https://docs.run.ai/latest/Researcher/scheduling/gpu-memory-fractions/
- NVIDIA blog: open-sourcing the Run:ai scheduler (KAI) — https://developer.nvidia.com/blog/
- NVIDIA GPU Operator (prerequisite) — https://docs.nvidia.com/datacenter/cloud-native/gpu-operator/latest/index.html

### Related Technologies

- **KAI Scheduler** — the open-sourced Run:ai scheduling core.
- **Kueue** — Kubernetes-native job queueing for batch workloads.
- **Volcano** — batch scheduler with gang scheduling for HPC/AI on Kubernetes.
- **YuniKorn** — universal resource scheduler with hierarchical queues.
- **NVIDIA GPU Operator / device plugin** — the GPU plumbing Run:ai builds on.